# 📖 Notebook 1: DNS and Load Balancing

📖 **Source**: [Hello Interview – Networking Essentials](https://www.hellointerview.com/learn/system-design/core-concepts/networking-essentials)

When you type `google.com` into your browser, how does your computer know which server to talk to? And when Google has *thousands* of servers, how does your request end up at the right one? The answer is **DNS** (for finding servers) and **Load Balancing** (for choosing which one).

## Learning Objectives

By the end of this notebook, you'll understand:
- How DNS translates domain names to IP addresses
- How DNS itself acts as a simple form of load balancing
- What a reverse proxy / load balancer does (using nginx)
- The difference between round-robin, least-connections, IP-hash, and weighted algorithms
- How health checks keep your system reliable

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/networking-essentials
docker-compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import socket
import requests
import time
from collections import Counter

# Base URL for our nginx load balancer running in Docker
NGINX_URL = "http://localhost:8080"

# Quick check that the infrastructure is running
try:
    r = requests.get(f"{NGINX_URL}/nginx-health", timeout=3)
    print(f"✅ Nginx is up: {r.text.strip()}")
except requests.ConnectionError:
    print("❌ Nginx is not running. Start it with: docker-compose up -d --build")

---
## Part 1: How DNS Works

**DNS** (Domain Name System) is like the phone book of the internet. Humans remember names (`google.com`), but computers communicate using IP addresses (`142.250.80.46`).

### The DNS Hierarchy

DNS is a **distributed, hierarchical** system:

```
You type google.com
    ↓
Your computer checks its local cache
    ↓ (cache miss)
Asks your ISP's DNS resolver
    ↓ (cache miss)
Asks a Root DNS server → "Try the .com servers"
    ↓
Asks the .com TLD server → "google.com is managed by ns1.google.com"
    ↓
Asks Google's authoritative DNS → "142.250.80.46"
    ↓
Your browser connects to 142.250.80.46
```

Let's see this in action!

In [ ]:
# === DNS Lookup with Python ===
# socket.getaddrinfo() does the same thing your browser does:
# it asks the OS to resolve a domain name to IP addresses.

domains = ["google.com", "github.com", "example.com"]

for domain in domains:
    # AF_INET = IPv4, SOCK_STREAM = TCP
    results = socket.getaddrinfo(domain, 80, socket.AF_INET, socket.SOCK_STREAM)
    # Each result is (family, type, proto, canonname, sockaddr)
    ips = list(set(r[4][0] for r in results))
    print(f"{domain:20s} → {ips}")

In [ ]:
# === DNS as Load Balancing ===
# Large sites return MULTIPLE IP addresses.
# Your client picks one — effectively distributing load across servers!
#
# Run this cell several times — you may see the order of IPs change.
# This rotation is called "DNS round-robin".

domain = "google.com"
print(f"Looking up {domain} three times:\n")

for i in range(3):
    results = socket.getaddrinfo(domain, 443, socket.AF_INET, socket.SOCK_STREAM)
    ips = [r[4][0] for r in results]
    print(f"  Attempt {i+1}: {ips}")
    time.sleep(0.5)

print("\n💡 DNS returns multiple IPs so clients spread across different servers.")
print("   This is 'client-side load balancing' — the simplest form of LB!")

### DNS TTL (Time To Live)

DNS records have a **TTL** — how long the answer can be cached. This matters because:
- **Short TTL** (e.g., 60 seconds) → changes propagate fast, but more DNS lookups
- **Long TTL** (e.g., 24 hours) → fewer lookups, but changes take time to reach everyone

In system design, if you need to quickly redirect traffic (e.g., during a failover), you want a **short TTL**.

---
## Part 2: Load Balancing with Nginx

DNS load balancing is simple but limited — it can't check if a server is healthy, and clients cache results. For real traffic management, we use a **dedicated load balancer**.

Our Docker setup has:
- **3 Flask backends** (`backend1`, `backend2`, `backend3`) — identical servers
- **1 Nginx** — sits in front and decides which backend gets each request

```
Client → Nginx (load balancer) → backend1
                                → backend2
                                → backend3
```

This is a **Layer 7 (L7) load balancer** because nginx understands HTTP. It can inspect URLs, headers, and cookies to make routing decisions.

### Algorithm 1: Round-Robin (Default)

The simplest algorithm: requests go to servers **in order**, cycling through them.

```
Request 1 → backend1
Request 2 → backend2
Request 3 → backend3
Request 4 → backend1  (back to the start)
```

In [ ]:
# === Round-Robin Load Balancing ===
# Send 12 requests to nginx and see which backend handles each one.

print("Round-Robin — sending 12 requests to nginx:\n")

servers_hit = []
for i in range(12):
    r = requests.get(NGINX_URL)
    data = r.json()
    server = data["server"]
    servers_hit.append(server)
    print(f"  Request {i+1:2d} → {server}")

print(f"\n📊 Distribution: {dict(Counter(servers_hit))}")
print("💡 Notice how requests cycle through backends evenly!")

### Algorithm 2: Least Connections

Send each request to the server with the **fewest active connections**. This is smarter than round-robin when some requests take longer than others.

```
backend1: 5 active connections
backend2: 2 active connections  ← next request goes here
backend3: 4 active connections
```

In [ ]:
# === Least Connections ===
# We send many requests to the /least-conn/ path which uses least_conn.

import concurrent.futures

def make_request(url):
    """Make a request and return which server handled it."""
    r = requests.get(url)
    return r.json()["server"]

print("Least Connections — sending 15 requests:\n")

servers_hit = []
for i in range(15):
    server = make_request(f"{NGINX_URL}/least-conn/")
    servers_hit.append(server)
    print(f"  Request {i+1:2d} → {server}")

print(f"\n📊 Distribution: {dict(Counter(servers_hit))}")
print("💡 With quick sequential requests, this looks similar to round-robin.")
print("   The difference shows up when some requests are SLOW (see next cell).")

In [ ]:
# === Least Connections with Slow Requests ===
# First, start some slow requests in the background.
# Then send fast requests — they should avoid the busy server.

print("Simulating: 3 slow requests (2s each) + 9 fast requests in parallel\n")

results = []

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as pool:
    # Start 3 slow requests (these will occupy backends)
    slow_futures = [
        pool.submit(make_request, f"{NGINX_URL}/least-conn/slow?delay=2")
        for _ in range(3)
    ]
    time.sleep(0.3)  # let slow requests connect

    # Now send fast requests — least_conn should route around busy servers
    fast_futures = [
        pool.submit(make_request, f"{NGINX_URL}/least-conn/")
        for _ in range(9)
    ]

    slow_results = [f.result() for f in slow_futures]
    fast_results = [f.result() for f in fast_futures]

print(f"Slow requests handled by: {slow_results}")
print(f"Fast requests handled by: {fast_results}")
print(f"\n📊 Fast request distribution: {dict(Counter(fast_results))}")
print("💡 Least-connections avoids servers that are busy with slow requests!")

### Algorithm 3: IP Hash (Sticky Sessions)

The same client IP **always** goes to the same backend. This is useful when backends store session state in memory.

```
Client 10.0.0.1 → always backend2
Client 10.0.0.2 → always backend1
Client 10.0.0.3 → always backend3
```

In [ ]:
# === IP Hash (Sticky Sessions) ===
# Every request from our machine should go to the SAME backend.

print("IP Hash — sending 10 requests (all from the same IP):\n")

servers_hit = []
for i in range(10):
    r = requests.get(f"{NGINX_URL}/ip-hash/")
    server = r.json()["server"]
    servers_hit.append(server)
    print(f"  Request {i+1:2d} → {server}")

print(f"\n📊 Distribution: {dict(Counter(servers_hit))}")
print("💡 All requests go to the same backend — that's 'sticky sessions'!")
print("   Useful when servers store session data in memory.")

### Algorithm 4: Weighted Round-Robin

Give more traffic to more powerful servers. In our config, `backend1` has weight 3, while `backend2` and `backend3` have weight 1. So `backend1` gets 3× the traffic.

In [ ]:
# === Weighted Round-Robin ===
# backend1 has weight=3, backend2 and backend3 have weight=1.
# So out of every 5 requests, backend1 should get 3.

print("Weighted Round-Robin — sending 20 requests:\n")

servers_hit = []
for i in range(20):
    r = requests.get(f"{NGINX_URL}/weighted/")
    server = r.json()["server"]
    servers_hit.append(server)

counts = Counter(servers_hit)
total = sum(counts.values())

print("📊 Distribution:")
for server in sorted(counts):
    pct = counts[server] / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {server}: {counts[server]:2d} requests ({pct:.0f}%) {bar}")

print("\n💡 backend1 gets ~60% of traffic (weight 3/5).")
print("   Use this when some servers are more powerful than others.")

---
## Part 3: Health Checks

Load balancers don't just distribute traffic — they also **monitor backend health**. If a server crashes, the load balancer stops sending it traffic.

Nginx checks backends passively by default: if a backend returns an error or times out, nginx marks it as "down" temporarily.

Let's simulate this by stopping one backend:

In [ ]:
import subprocess

# === Health Checks: Stop a backend and watch nginx adapt ===

print("BEFORE stopping backend2:")
servers_before = []
for i in range(9):
    r = requests.get(NGINX_URL)
    servers_before.append(r.json()["server"])
print(f"  Servers hit: {dict(Counter(servers_before))}\n")

# Stop backend2
print("⏸️  Stopping backend2...")
subprocess.run(["docker", "stop", "net-backend2"], capture_output=True)
time.sleep(2)  # give nginx a moment to detect the failure

print("\nAFTER stopping backend2:")
servers_after = []
for i in range(9):
    r = requests.get(NGINX_URL)
    servers_after.append(r.json()["server"])
print(f"  Servers hit: {dict(Counter(servers_after))}")
print("\n💡 Nginx automatically routes around the dead server!")

# Restart backend2
print("\n▶️  Restarting backend2...")
subprocess.run(["docker", "start", "net-backend2"], capture_output=True)
time.sleep(2)

print("\nAFTER restarting backend2:")
servers_restarted = []
for i in range(9):
    r = requests.get(NGINX_URL)
    servers_restarted.append(r.json()["server"])
print(f"  Servers hit: {dict(Counter(servers_restarted))}")
print("\n💡 Traffic includes backend2 again — automatic recovery!")

---
## Part 4: L4 vs L7 Load Balancers

| Feature | Layer 4 (Transport) | Layer 7 (Application) |
|---------|--------------------|-----------------------|
| Inspects | IP addresses, ports | HTTP headers, URLs, cookies |
| Speed | Very fast | Slightly slower |
| Routing | Random / hash-based | Content-based (URL paths, headers) |
| Best for | WebSockets, raw TCP | REST APIs, HTTP services |
| Example | AWS NLB, HAProxy (TCP mode) | Nginx, AWS ALB, HAProxy (HTTP mode) |

Our nginx setup is an **L7 load balancer** — it reads the URL path to decide which upstream group to use:
- `/` → round-robin
- `/least-conn/` → least connections
- `/ip-hash/` → sticky sessions
- `/weighted/` → weighted distribution

An L4 load balancer couldn't do this — it would just forward raw TCP connections without knowing the URL.

In [ ]:
# === Proof that nginx inspects HTTP content (L7) ===
# Look at the headers the backend receives — nginx adds its own.

r = requests.get(f"{NGINX_URL}/headers")
headers = r.json()["headers"]

print("Headers received by the backend:\n")
for key, value in headers.items():
    marker = " ← added by nginx" if key.startswith("X-") else ""
    print(f"  {key}: {value}{marker}")

print("\n💡 Nginx adds X-Real-IP, X-Forwarded-For, and X-Load-Balancer headers.")
print("   An L4 LB wouldn't be able to add these — it doesn't understand HTTP.")

---
## 🎓 Key Takeaways

1. **DNS** translates names to IPs and acts as basic client-side load balancing
2. **Dedicated load balancers** (like nginx) give you more control over traffic distribution
3. **Round-robin** is the simplest and default algorithm — great for stateless services
4. **Least-connections** is better when request durations vary
5. **IP hash** provides sticky sessions for stateful backends
6. **Health checks** automatically route around failed servers
7. **L7 LBs** understand HTTP and can route by URL/headers; **L4 LBs** are faster but simpler

### Interview Tips
- Default to an **L7 load balancer** for HTTP/REST traffic
- Use **L4** for WebSocket or raw TCP connections
- Mention **DNS** for avoiding a single point of failure with your load balancers
- **Health checks** are essential — always mention them when discussing reliability